In [ ]:
## 1. Setup & Data Loading
import pandas as pd 
import numpy as np 
import plotly.express as px 
import plotly.figure_factory as ff
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from plotly.subplots import make_subplots
warnings.filterwarnings("ignore")


df = pd.read_csv("Loan_default.csv")
df

#Missing Values in the DataSets 
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending = False)
missing
"""There is no missing Values in the Data Sets"""

print("\nShape of Data (Rows, Columns)")
print(df.shape)

print("\nInforamation of the Data")
print(df.info())

print("\nStatistics of the Data")
print(df.describe())

In [ ]:
## 2. Initial Data Sanity Check

df.dtypes
df["LoanAmount"].max()

loan_amount_greater = df[(df["LoanAmount"] > 200000) & (df["Age"] > 50) & (df["Income"] < 20000)]
loan_amount_greater = loan_amount_greater.value_counts()
print(loan_amount_greater)

Loan_purpose_dist = (df["LoanPurpose"].value_counts())
Loan_purpose_dist = pd.DataFrame(Loan_purpose_dist)


fig = make_subplots(rows=2, cols=2, subplot_titles= ["CreditScore vs Default", "DTIRatio vs Default", "InterestRate vs Defaul", "Income vs Default"])

fig.add_box(
    x = df["Default"],
    y= df["CreditScore"],
    row= 1, col=1,
    name= "Default Vs CreditScore"
)

fig.add_box(
    x= df["Default"],
    y=df["DTIRatio"],
    row = 1, col = 2, 
    name= "DTIRatio vs Defaul"
)

fig.add_box(
    x=df["Default"],
    y = df["InterestRate"],
    row=2, col=1,
    name= "InterestRate vs Default"
)

fig.add_box(
    x=df["Default"],
    y = df["Income"],
    row=2, col=2
)

fig.show()



In [ ]:
## 3. Default Rate by Categorical Features

def default_rate(column):
    default_rate = df.groupby(column)["Default"].mean().mul(100).sort_values(ascending = False).round(3)
    return default_rate

print("Default Rate for the Education")
default_rate_education = default_rate("Education")
print(default_rate_education)

print("\nDefault Rate for the Employment Type")
default_rate_employment = default_rate("EmploymentType")
print(default_rate_employment)

print("\nDefault Rate for the Mariage Status")
default_rate_marital = default_rate("MaritalStatus")
print(default_rate_marital)

print("\nDefault Rate for the Mortagage Status")
default_rate_mortage = default_rate("HasMortgage")
print(default_rate_mortage)

print("\nDefault Rate for the Dependent(yes/not)")
default_rate_dependence = default_rate("HasDependents")
print(default_rate_dependence)

print("\nDefault Rate for the Loan Purpose")
default_rate_purpose = default_rate("LoanPurpose")
print(default_rate_purpose)

print("\nDefault Rate for the CoSigner")
default_rate_Cosigner = default_rate("HasCoSigner")
print(default_rate_Cosigner)



In [ ]:
## 4. Visualizing Default Rates (All Categorical Features Together)

fig2 = make_subplots(
    rows= 4, cols= 2, subplot_titles= [
        "Default Rate for the Education",
        "Default Rate for the Employment Type",
        "Default Rate for the Mariage Status",
        "Default Rate for the Mortagage Status",
        "Default Rate for the Dependent(yes/not)",
        "Default Rate for the Loan Purpose",
        "Default Rate for the CoSigner"
    ]
)

def plotly_bar(default_rate_, row, col):
    fig2.add_bar(
        x = default_rate_.index,
        y = default_rate_.values,
        row = row, 
        col = col
    )
    
plotly_bar(default_rate_education,1,1)
plotly_bar(default_rate_employment,1,2)
plotly_bar(default_rate_marital,2,1)
plotly_bar(default_rate_mortage, 2,2)
plotly_bar(default_rate_dependence,3,1)
plotly_bar(default_rate_purpose,3,2)
plotly_bar(default_rate_Cosigner,4,2)

fig2.update_layout(height=1000, width=900, title_text="7 Subplots in a 4x2 Grid")
fig2.show()

In [ ]:
## 5. Correlation Analysis (Numeric Features)
numeric_only = df.select_dtypes(include= "number").corr().round(3).abs()

fig3 = px.imshow(
    numeric_only,
    text_auto=".2f",
    aspect="auto",
    color_continuous_scale="RdBu",
    range_color=[-1,1],
    title="Correlation Heatmap Matrix"
)

fig3.show()


In [ ]:
## 6. Feature Interaction Check (Age vs Income)
df_1k = df.head(500)

fig4 = px.scatter(df_1k, x = df_1k["Age"], y= df_1k["Income"], color="Age", size="Income")
fig4.show()

In [ ]:
## 7. Feature Engineering — Interaction Features
df["Risk Factor Combine"] = df["DTIRatio"]*df["CreditScore"]
df["Loan burden"] = df["LoanAmount"]/df["Income"]
df["Total Interest burden"] = df["InterestRate"]*df["LoanTerm"]
df["Income Stability proxy"] = df["Income"]/df["MonthsEmployed"]

In [ ]:
## 8. Feature Selection — Dropping Low-Value Columns
print(df["LoanAmount"].describe())
print(df["LoanAmount"].median())
df = df.drop(columns= ["LoanID", "LoanTerm", "Risk Factor Combine", "Total Interest burden", "Income Stability proxy"], axis=1)
print(df.shape)
df_20k = df.head(100000)
df_20k.shape

In [ ]:
## 9. Model 1 — Random Forest (Baseline with Class Balancing)
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import RandomizedSearchCV


df_encoded = pd.get_dummies(df, columns=["Education", "EmploymentType", "MaritalStatus", "HasMortgage", 
                                     "HasDependents", "LoanPurpose", "HasCoSigner"], drop_first= True)

X = df_encoded.drop(columns=["Default"], axis = 1)
y = df_encoded["Default"]

X_train, X_test, y_train, y_test = train_test_split(
    X,y,
    test_size=0.2,
    random_state=42,
    stratify= y 
)

model = RandomForestClassifier(
    n_estimators=200,
    max_depth= 5,
    max_features="log2",
    min_samples_split= 2,
    min_samples_leaf= 2,
    random_state=42,
    n_jobs= -1,
    class_weight="balanced"
)

model.fit(X_train, y_train)
print(df_encoded.shape)

y_pred = model.predict(X_test)
print(f"Accuracy Score = {accuracy_score(y_test, y_pred)}")
print(f"{classification_report(y_test, y_pred)}")
print(f"AUC Score {roc_auc_score(y_test, y_pred)}")


In [ ]:
## 10. Model 2 — XGBoost with Hyperparameter Tuning
from xgboost import XGBClassifier

model_2 = XGBClassifier(
    use_label_encoder = False,
    eval_metric = "logloss",
    random_state = 42,
    scale_pos_weight = 7.6,
)

param_dist = {
    "n_estimators" : [50, 100, 200, 400, 500],
    "max_depth" : [3,5,7, None],
    "learning_rate" : [00.1, 00.5, 0.1, 0.2, 0.3],
    "subsample" : [0.6, 0.7, 0.8, 0.9, 1.0],
    "gamma" : [0.1, 0.2, 0.3],
    "reg_lambda" : [0.1,1, 5]
    }

random_search2 = RandomizedSearchCV(
    estimator= model_2,
    n_iter=20, 
    cv= 5,
    param_distributions=param_dist,
    n_jobs= -1,
    scoring="precision",
    verbose= 2
)

random_search2.fit(X_train, y_train)

best_model = random_search2.best_estimator_
y_pred2 = best_model.predict(X_test)

print(f"Best Parameters : {random_search2.best_params_}")
print(f"Best precision Score : {random_search2.best_score_}")
print(f"{classification_report(y_test, y_pred2)}")
print(f"{accuracy_score(y_test, y_pred2)}")
print(f"AUC score {roc_auc_score(y_test,y_pred2)}")

In [ ]:
## 11. Feature Importance & Interpretation
importance = pd.DataFrame({
    "Feature" : X.columns,
    "Importance": best_model.feature_importances_
}).sort_values(by = "Importance",ascending=False)

print(importance)

In [ ]:
# ## 12. Export Model for Deployment

# import joblib

# joblib.dump(best_model, "Loan_Default_model.pkl")

# model_columns = X.columns.tolist()
# joblib.dump(model_columns, "model_columns.pkl")

In [ ]:
## 13. Inference Sanity Check

# Apply the same feature engineering and encoding used for training.
input_data = pd.DataFrame([{
    'Age': 39,
    'Income': 60000,
    'LoanAmount': 15000,
    'CreditScore': 650,
    'MonthsEmployed': 20,
    'NumCreditLines': 3,
    'InterestRate': 12.50,
    'DTIRatio': 0.35,
    'Education': "High School",
    'EmploymentType': "Full-time",
    'MaritalStatus': "Single",
    'HasMortgage': "No",  # Added missing feature
    'HasDependents': "Yes",
    'LoanPurpose': "Education",
    'HasCoSigner': "Yes"
}])

input_data["Loan burden"] = input_data["LoanAmount"] / input_data["Income"]
input_encoded = pd.get_dummies(
    input_data,
    columns=["Education", "EmploymentType", "MaritalStatus", "HasMortgage",
             "HasDependents", "LoanPurpose", "HasCoSigner"],
    drop_first=True,
)
input_final = input_encoded.reindex(columns=X.columns, fill_value=0)
probability = best_model.predict_proba(input_final)[0, 1]
prediction = best_model.predict(input_final)[0]
print(f"Prediction: {prediction}; probability of default: {probability:.2%}")